## Energy efficiency

### We start with a matmul kernel to tune, similar to the previous section

In [ ]:
import numpy as np

import kernel_tuner as kt

In [ ]:
kernel_name = "matmul"
kernel_code = """
__global__ void matmul(int n, float* A, float* B, float* C) {
  int j = blockIdx.x * blockDim.x + threadIdx.x;
  int i = blockIdx.y * blockDim.y + threadIdx.y;

  if (i < n && j < n) {
    float sum = 0;
    for (int k = 0; k < n; k++) {
      sum += A[i * n + k] * B[k * n + j];
    }
    C[i * n + j] += sum;
  }
}
"""

tune_params = dict()
tune_params["block_size_x"] = [8, 16, 32, 64, 128]
tune_params["block_size_y"] = [1, 2, 4, 8, 16]

In [ ]:
n = 2048
A = np.random.rand(n, n).astype("float32")
B = np.random.rand(n, n).astype("float32")
C_expected = np.matmul(A, B)
C = np.zeros_like(C_expected)

In [ ]:
problem_size = (n, n)
arguments = [np.int32(n), A, B, C]
answer = [None, None, None, C_expected]

In [ ]:
# Run the tuner!
kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    lang="nvcuda",
);

### Performance

Run time is just one measure of performance. For a smaller amount of data, the runtime will be shorter. Is is fair to say that performance is then better? A fairer performance number may be the amount of operations executed per second. In this case, we are talking about a floating-point operation, or FLOPs. Examples are a single multiplication or addition operation. Note that the data type matters: A GPU typically has different maximum performance in single precision (float32) than double precision (float64).

The performance is then expressed as the number of FLOPs per second, typically writen as just FLOPs but to avoid confusion we will use FLOP/s here.

For a matrix-matrix multiplication of a MxK matrix by a KxN matrix, we need to execute MxN vector dot products, each of K elements long. That means in total MxNxK multiply-add (FMA) operations need to be performed, for a total of 2xMxNxK FLOPs.

Let's add the FLOP/s measurement to our tuning. Because GPUs are so fast, we will scale the number and use giga-FLOP/s, or GFLOP/s.

In [ ]:
gflop = 2 * n * n * n * 1e-9

def get_gflops(result):
    return gflop / (result["time"] * 1e-3)

metrics = {}
metrics["GFLOP/s"] = get_gflops

kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    lang="nvcuda",
    metrics=metrics
);

### Power measurements

GPUs, as well as CPUs, contain sensors that continuously measure their power consumption. These sensors can be accessed through vendor-specific libraries. For NVIDIA, that library is NVML. For AMD, there is AMD SMI. These libraries share some functionality with the `nvidia-smi` and `amd-smi` command-line tools.

Kernel Tuner has a way to use NVML through an _observer_. Let's try this out and measure the GPU power.

In [ ]:
from kernel_tuner.observers.nvml import NVMLObserver

observer = NVMLObserver(["nvml_energy", "nvml_power"])

def get_power(result):
    return result["nvml_power"]

metrics["Power(W)"] = get_power

kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    lang="nvcuda",
    metrics=metrics,
    observers=[observer]
);

### Energy effiency

There are many ways to define energy efficiency. The definition commonly used in GPU computing is the performance in GFLOP/s divided by the average GPU power in Watt during the code execution, so we end up with GFLOP/s/W. However, power is the same as energy per second: 1 W = 1 J / s. A GFLOP/s/W is thus equivalent to a GFLOP/J, in other words: the number of floating point operations per Joule of energy. We will add this energy efficiency measurement to our tuning. NVML provides the energy in Joule in addition to the power, so we can directly calculate the GFLOP/J.

In [ ]:
def get_gflops_per_joule(result):
    return gflop / result["nvml_energy"]

metrics["GFLOP/J"] = get_gflops_per_joule

tuning_results, env = kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    lang="nvcuda",
    metrics=metrics,
    observers=[observer]
);

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

time = [result["GFLOP/s"] for result in tuning_results]
energy = [result["GFLOP/J"] for result in tuning_results]

plt.scatter(energy, time)
plt.xlabel("GFLOP/J")
plt.ylabel("GFLOP/s")
plt.show()

In this plot of performance vs efficiency we can see there is as strong correlation between the two. However, this is not always the case, especially as one expands the tuning search space. It is also possible (with admin rights) to change the clock frequency of the GPU. Often, a slighly lower than maximum clock freuqency results in a higher energy efficiency at only a slight loss of performance.

### How about other sensors / vendors?

As mentioned, other vendors have different libraries to access their sensors. There are libraries for other types of GPUs, but also for e.g. CPUs. One library with a unified interface to several such sensors is [PMT](https://git.astron.nl/RD/pmt). It supports NVML, AMD-SMI, and a few other. PMT is supported by Kernel Tuner as well.